# Splits temporais, janelas e vazamento

Este notebook explica por que os experimentos RioNowcast separam treino, validacao e teste por anos inteiros. A mesma regra vale para qualquer nova fonte de dados.

## Protocolo multianual

| Split | Anos | Papel |
|---|---|---|
| Treino | 2012-2021 | Ajusta os parametros do modelo |
| Validacao | 2022 | Early stopping e selecao de configuracao |
| Teste | 2023-2024 | Avaliacao final, usada uma unica vez |

Separar por anos evita que janelas de uma mesma sequencia temporal aparecam em treino e teste.

In [ ]:
from __future__ import annotations

import pandas as pd

T_IN = 5
T_OUT = 5
STRIDE = 5
timestamps = pd.date_range('2024-01-01 00:00', periods=24, freq='15min', tz='UTC')

def describe_window(start: int) -> dict:
    input_indexes = list(range(start, start + T_IN))
    target_indexes = list(range(start + T_IN, start + T_IN + T_OUT))
    return {
        'inicio': start,
        'entrada_frames': input_indexes,
        'entrada_inicio': timestamps[input_indexes[0]],
        'entrada_fim': timestamps[input_indexes[-1]],
        'target_frames': target_indexes,
        'target_inicio': timestamps[target_indexes[0]],
        'target_fim': timestamps[target_indexes[-1]],
    }

windows = pd.DataFrame([describe_window(start) for start in range(0, 15, STRIDE)])
display(windows)

Com `t_in=5`, `t_out=5` e `stride=5`, a previsao de uma janela ocupa o periodo que servira como entrada da proxima. Isso e aceitavel dentro de um mesmo split cronologico; nao e aceitavel repartir essas janelas aleatoriamente entre treino e teste.

## Exemplo de vazamento com split aleatorio

Quando `stride=1`, janelas adjacentes compartilham quase todos os frames. Se uma for para treino e outra para teste, o teste deixa de representar dados futuros independentes.

In [ ]:
window_a = describe_window(0)
window_b = describe_window(1)

shared = sorted(set(window_a['entrada_frames'] + window_a['target_frames']) & set(window_b['entrada_frames'] + window_b['target_frames']))
display(pd.DataFrame([window_a, window_b], index=['janela A', 'janela B']))
print('Frames compartilhados entre as duas janelas:', shared)
print('Conclusao: um split aleatorio de janelas introduziria vazamento.')

## O que `step` e `stride` controlam

- `step=5`: quantidade de passos futuros prevista pelo STConvS2S.
- `t_in=5`: quantidade de passos passados fornecida ao modelo.
- `stride=5`: salto entre inicios de janelas consecutivas no dataset.

Com resolucao de 15 minutos, cinco passos correspondem a 75 minutos. O dataset nunca cria uma janela que atravesse a fronteira entre anos, pois cada ano e carregado separadamente.

In [ ]:
duration = pd.Timedelta(minutes=15 * T_IN)
print('Contexto de entrada:', duration)
print('Horizonte total de previsao:', pd.Timedelta(minutes=15 * T_OUT))
print('Salto entre janelas:', pd.Timedelta(minutes=15 * STRIDE))

## Regras para novas features

Uma feature adicional, como uma banda do GOES ou chuva defasada de estacoes, deve obedecer ao mesmo recorte temporal da entrada:

```text
features de entrada: s : s+5
targets avaliados:       s+5 : s+10
```

Nao e permitido usar imagem de satelite, observacao de estacao, normalizacao ou estatistica calculada depois de `s+4` para prever essa janela. Parametros de normalizacao tambem devem ser ajustados exclusivamente no treino.

## Checklist de um experimento temporal

1. Anos de treino, validacao e teste nao se sobrepoem.
2. Todas as fontes possuem timestamps alinhados a grade de 15 minutos.
3. Entradas usam apenas informacao disponivel antes do primeiro horizonte previsto.
4. Pesos, normalizacao e hiperparametros sao escolhidos sem consultar o teste.
5. A comparacao entre modelos usa os mesmos anos, targets, mascara e numero de observacoes de teste.